# LOGIN

In [15]:
import re
from selenium import webdriver
from helper_functions.selenium_helper import handywrapper

class login:
    """class to handle the login process, it takes the driver as input and returns the driver after login"""

    def open_and_login(self):
        """function to open the website and login to the website, it takes the driver as input and returns the driver after login"""
        # opening the website and logging in to the website
        driver=webdriver.Chrome()
        driver.implicitly_wait(10)
        driver.get("https://hcm41.sapsf.com")
        wrapper = handywrapper(driver)

        #entering company ID
        wrapper.send_keys_to_element("xpath", "//input[@placeholder='Enter Company ID']","codebotforD")

        #press continue button
        wrapper.click_element('XPATH',"//bdi[contains(text(),'Continue')]")

        #entering username and password
        #email
        wrapper.send_keys_to_element('XPATH',"//input[@placeholder='Email or User Name']", "sfadmin")

        #password
        wrapper.send_keys_to_element('XPATH',"//input[@placeholder='Password']", "Sfcode123")

        #clicking on continue button
        wrapper.click_element('XPATH',"//div[contains(text(),'Continue')]")

        #accepting the data privacy policy
        wrapper.click_element('XPATH',"//button[contains(text(),'Accept')]")

        #returning driver for further use in other modules
        return driver



In [16]:
login=login()
driver=login.open_and_login()

# SAP SCRAPPING

In [ ]:
from helper_functions.selenium_helper import handywrapper
from Net2apps_tasks.Main_Task.ratingscale_designer.reverse.data_model_rating_scale import Processing_Model,Rating_Scale_Model,Scores_Model

class module_reverse:
    def extract_data(self,driver):
        wrapper = handywrapper(driver)

        url = driver.current_url
        scrub_id = re.search(r'_s\.crb=([^&]+)', url).group(1)
        new_url = f"https://hcm41.sapsf.com/acme?fbacme_o=admin&pess_old_admin=true&ap_param_action=form_rating_scale&itrModule=talent&_s.crb={scrub_id}"
        driver.get(new_url)
        
        #scrapping all the instances of rating scale designer
        instance_name=wrapper.get_elements_text('XPATH',"//a[contains(@id,'link') and @class='fd-link fd-link--compact']")

        data_list=[]
        # iterating through all the instances and scrapping the data
        for i, instance in enumerate(instance_name):
            print(f"============= Instance {i+1}: {instance} =============")
            #clicking on the instance
            wrapper.click_element('LINK_TEXT', instance)
            # apply exception handling to avoid the error if the OK button is not available
            try:
                #fetching the name of the rating scale
                name=wrapper.element_attribute('ID', "48:_txtFld", "value")
            except:
                 # clicking okay if avaialable
                wrapper.click_element('XPATH',"//button[contains(text(),'OK')]")
                #fetching the name of the rating scale
                name=wrapper.element_attribute('ID', "48:_txtFld", "value")

            description=wrapper.element_attribute('ID', "50:_txtArea", "value")

            # all score ,labels and descriptions 
            scores=wrapper.find_elements('XPATH',"//input[@size='7']")
            labels=wrapper.find_elements('XPATH',"//input[@size='34']")
            option_description=wrapper.find_elements('XPATH',"//textarea[@cols='42']")
            DMS1=Processing_Model()
            DMS2=Rating_Scale_Model()

            DMS1.item_id=i+1
            DMS1.processing_status="Pending"
            DMS1.processing_rating_scale=name

            DMS2.item_id=i+1
            DMS2.rating_scale_name=name
            DMS2.rating_scale_description=description

            #fetching the description of the rating scale
            for j in range(len(scores)):
                DMS3=Scores_Model()
                DMS3.item_id = i+1
                DMS3.rating_scale = name
                DMS3.option_score = scores[j].get_attribute("value")
                DMS3.option_label = labels[j].get_attribute("value")
                DMS3.option_description = option_description[j].text
                data_list.append([DMS1, DMS2, DMS3])
            
            # refreshing the page to avoid stale element reference error
            driver.refresh()

            # returning the all_data list to the calling function
        return data_list

In [ ]:
reverse=module_reverse()
all_data=reverse.extract_data(driver)

============= Instance 1: ztest new object_6 =============
============= Instance 2: ztest new object_12 =============
============= Instance 3: ztest new object_18 =============
============= Instance 4: ztest new object_22 =============
============= Instance 5: ztest new object_28 =============
============= Instance 6: ztest new object_34 =============
============= Instance 7: ztest new object_7 =============
============= Instance 8: ztest new object_11 =============
============= Instance 9: ztest new object_15 =============
============= Instance 10: ztest new object_19 =============
============= Instance 11: ztest new object_23 =============
============= Instance 12: ztest new object_27 =============
============= Instance 13: ztest new object_31 =============
============= Instance 14: ztest new object_35 =============
============= Instance 15: ztest new object_666 =============
============= Instance 16: ztest new object_5 =============
============= Instance 17: ztest ne

# Fill Sheet model

In [ ]:
from helper_functions.gspread_healper import GspreadSheetHelper
class fill_module_sheet:
    """class to fill the data into the google sheet"""

    #making object of  GspreadSheetHelper
    GSH=GspreadSheetHelper()
    worksheet = GSH.getsheet("Rating Scale")
    # Clearing the sheet before filling the data to avoid duplication
    GSH.clear_sheet(worksheet)

    def processing_fill_controller(self,models):
        """function to fill the data into the google sheet of Section 1"""
        rows=[]
        for  model in models:
            rows.append([model[0].item_id, 
                        model[0].processing_status,
                        model[0].processing_rating_scale])

        self.GSH.update_sheet(self.worksheet,'A','C',rows)
        
        

    def rating_scale_fill_controller2(self,models):
        """function to fill the data into the google sheet of Section 2"""
        rows=[]
        for model in models:
            rows.append([model[1].item_id,
                        model[1].rating_scale_name,
                        model[1].rating_scale_description])

        self.GSH.update_sheet(self.worksheet,'E','G',rows)



    def scores_fill_controller3(self,models):
        """function to fill the data into the google sheet of Section 3"""

        rows=[]
        for model in models:
            rows.append([model[2].item_id,
                        model[2].rating_scale,
                        model[2].option_score,
                        model[2].option_label,
                        model[2].option_description])
  
        self.GSH.update_sheet(self.worksheet,'I','M',rows)


In [ ]:
fms=fill_module_sheet()
fms.processing_fill_controller(all_data)
fms.rating_scale_fill_controller2(all_data)
fms.scores_fill_controller3(all_data)

No data found in the sheet
Clearing data from row 4 to 500 in the worksheet.


# Data Validation

In [ ]:
from gspread_formatting import CellFormat, Color, format_cell_ranges
from helper_functions.gspread_healper import GspreadSheetHelper
from Net2apps_tasks.Main_Task.ratingscale_designer.reverse.controllers_rating_scale import loader_controller
from helper_functions.operation_helper import operation_helpers
class validation_model:

    # Selection of color for validation
    green = CellFormat(backgroundColor=Color(0.7, 1, 0.7))
    red = CellFormat(backgroundColor=Color(1,0.7,0.7))

    # creating object of GspreadSheetHelper, loader_controller and operation_helpers
    GSH=GspreadSheetHelper()
    loader=loader_controller()
    helper=operation_helpers()
    worksheet=GSH.getsheet("Rating Scale")
    
    def validate_processing_section(self,models):
        """function to validate the data in the google sheet of Section 1"""

        processing_section_data=self.loader.processing_loader_controller()
        formate=[]
        for i, row in enumerate(processing_section_data):
            exist, index = self.helper.check_scale_12section(processing_section_data, models, i)
            if exist:
                color = self.green if row.processing_rating_scale == models[index][0].processing_rating_scale else self.red
                formate.append((f"C{i+4}", color))
            else:

                formate.append((f"C{i+4}", self.red))

        format_cell_ranges(self.worksheet,formate)
        print("Validation Completed for Processing Section")


    def validate_rating_section(self,models):
        """function to validate the data in the google sheet of Section 2"""

        rating_section_data=self.loader.rating_scale_loader_controller()
        formate=[]
        for i, row in enumerate(rating_section_data):
            exist, index = self.helper.check_scale_12section(rating_section_data, models, i)
            if exist:
                color = self.green if row.rating_scale_name == models[index][1].rating_scale_name else self.red
                formate.append((f"F{i+4}", color))

                color = self.green if row.rating_scale_description == models[index][1].rating_scale_description else self.red
                formate.append((f"G{i+4}", color))
            else:
                formate.append((f"F{i+4}", self.red))
                formate.append((f"G{i+4}", self.red))

        format_cell_ranges(self.worksheet,formate)
        print("Validation Completed for Rating Scale Section")


    def validate_scores_section(self,models):
        """function to validate the data in the google sheet of Section 3"""

        scores_section_data=self.loader.scores_loader_controller()
        formate=[]
        for i, row in enumerate(scores_section_data):
            exist, index = self.helper.check_scale_scores(scores_section_data, models, i)

            if exist:
                color = self.green if row.rating_scale == models[index][2].rating_scale else self.red
                formate.append((f"J{i+4}", color))

                color = self.green if row.option_score == models[index][2].option_score else self.red
                formate.append((f"K{i+4}", color))

                color = self.green if row.option_label == models[index][2].option_label else self.red
                formate.append((f"L{i+4}", color))

                color = self.green if row.option_description == models[index][2].option_description else self.red
                formate.append((f"M{i+4}", color))
            else:
                formate.append((f"J{i+4}", self.red))
                formate.append((f"K{i+4}", self.red))
                formate.append((f"L{i+4}", self.red))
                formate.append((f"M{i+4}", self.red))

        format_cell_ranges(self.worksheet,formate)
        print("Validation Completed for Scores Section")

In [14]:
val=validation_model()
val.validate_processing_section(all_data)
val.validate_rating_section(all_data)
val.validate_scores_section(all_data)


Validation Completed for Processing Section
Validation Completed for Rating Scale Section
Validation Completed for Scores Section


# Automation

In [ ]:
from helper_functions.gspread_healper import GspreadSheetHelper
from helper_functions.selenium_helper import handywrapper
from helper_functions.operation_helper import operation_helpers
from Net2apps_tasks.Main_Task.ratingscale_designer.reverse.controllers_rating_scale import loader_controller
class Automation_model:
    


    def automation(self,models,driver):
        """function to control the automation process for the rating scale designer"""
        # creating object of GspreadSheetHelper, loader_controller, handywrapper and operation_helpers
        loader=loader_controller()
        GSH = GspreadSheetHelper()
        wrapper= handywrapper(driver)
        helper=operation_helpers()
        worksheet = GSH.getsheet("Rating Scale")

        status=[]
        rating_scale_section_data=loader.rating_scale_loader_controller()
        scores_section_data=loader.scores_loader_controller()
        all_matched=True
        for i in range(len(rating_scale_section_data)):
                
                exist, index = helper.check_scale_scores(scores_section_data, models, i)
                if exist:
                    # calling the function that checks the condition for validation if true then green else red and the location
                    all_matched=(rating_scale_section_data[i].rating_scale_name == models[index][1].rating_scale_name and 
                                rating_scale_section_data[i].rating_scale_description == models[index][1].rating_scale_description and
                                scores_section_data[i].option_score == models[index][2].option_score and 
                                scores_section_data[i].option_label == models[index][2].option_label and 
                                scores_section_data[i].option_description == models[index][2].option_description)
                    
                if not all_matched or exist==False:
                    helper.update_scale(rating_scale_section_data[i], scores_section_data, wrapper)
            
                status.append(["Processed"])

               
        self.GSH.update_sheet(worksheet,'B','B',status)
        print(f"Automation completed. Total instances processed: {len(status)}. Status updated in the sheet for locations.")


In [68]:
am=Automation_model()
am.automation(all_data,driver)

Number of Scores in the data list: 2, Number of Scores in the web page: 3
Deleting an option in the rating scale designer.
Updated the existing instance of rating scale: ztest new object_4


C:\Users\tesla\AppData\Local\Temp\ipykernel_8200\4200013910.py:73: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  self.worksheet.update(location, status)


Automation completed. Total instances processed: 185. Status updated in the sheet for locations.
